In [15]:

import numpy as np
import matplotlib.pyplot as plt

# Set environment variables
import os


os.environ["SCAL_TYPE"] = "complex"
os.environ["PRECISION"] = "single"
os.environ["MY_NUMBA_TARGET"] = "numba"

# Add cle_fun to PYTHON_PATH
import sys
sys.path.append("../../clonscal")

In [16]:
import numpy as np
from simulation.config import Config
from simulation.cl_simulation import ComplexLangevinSimulation
from src.obs_kernels import (
    n_moment_kernel, 
    dse_n_moment_kernel,
    abs_drift
)
from tqdm import tqdm
from simulation.gpu_handler import GPU_handler
from src.numba_target import use_cuda
import src.scal as scal
from src.utils import (
    update_histogram_complex, 
    update_histogram_real,
    gaussian_modified_density_drift_kernel, 
    noise_kernel_rotated,
    calculate_stats_complex
)

from src.numba_target import my_act_parallel_loop
from numba import cuda

define a configuration with desired parameters. see simulation.config.py for defaults. Based on the config, define a simulation object

In [17]:
config = Config(dims = [1], sigma=1, interaction=1, trajs = 2, steps = int(1e4), dt = 1e-3)
sim = ComplexLangevinSimulation(config)

Register observables into the object. This defines ObservableTracker instances

In [18]:
sim.register_observable('1_moment', obs_kernel=n_moment_kernel, const_param={"order" : 1},  langevin_history=True, thermal_time=1, auto_corr=1.0)
sim.register_observable('2_moment', obs_kernel=n_moment_kernel, const_param={"order" : 2},  langevin_history=True, thermal_time=1, auto_corr=1.0)

These can not be addressed via

In [19]:
sim.trackers

{'1_moment': <simulation.observables.ObservableTracker at 0x7f3f689a9840>,
 '2_moment': <simulation.observables.ObservableTracker at 0x7f3f689a8100>}

The trackers are independent from each other. The observable carries information about all the trajectories, see shape of the arrays

In [20]:
sim.trackers["1_moment"].__dict__

{'obs_name': '1_moment',
 'shape': (2,),
 'obs_kernel': CPUDispatcher(<function n_moment_kernel at 0x7f3f6fc331c0>),
 'langevin_history': True,
 'const_param': {'order': 1},
 'init_history_size': 10000,
 'thermal_time': 1,
 'auto_corr': 1.0,
 'order': 1,
 '_sim_instance': <simulation.cl_simulation.ComplexLangevinSimulation at 0x7f3f689a82b0>,
 'equilibrated_trajs': array([False, False]),
 'meas_time': array([-1., -1.], dtype=float32),
 'result': array([0.+0.j, 0.+0.j], dtype=complex64),
 'history_result': array([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
       dtype=complex64),
 'history_counter': array([0, 0, 0, ..., 0, 0, 0], dtype=int32),
 'history_meas_times': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'kernel_bridge': <src.utils.KernelBridge at 0x7f3f6881fbe0>,
 'stats': <src.utils.RollingStats at 0x7f3f6881c070>,
 'rolling_mean': array([0.+0.j, 0.+0.j], dtype=complex64),
 'rolling_sqr_mean_real': array([0., 0.], dtype=float32),
 'rolling_sqr_mean_imag': arr

Once a simulation steps is performed

In [21]:
sim.step()

the global degree of freedom sim.phi is updated according to the langevin equation. It stores the values for all the trajectories. 

In [22]:
sim.phi0

array([0.01472317+0.j, 0.06300053+0.j], dtype=complex64)

After a step, we have to decide whether observables are to be calculated, depending on tracker.thermal_time and tracker.auto_corr.  
Every trajectory is associated with a boolean that allows or blocks calculation. These are set to False in the beginning

In [23]:
sim.trackers["1_moment"].equilibrated_trajs

array([False, False])

We could call tracker.compute(), but all trajectories will be ignored. Their result is unchanged (zero)

In [24]:
sim.trackers["1_moment"].compute()
print(sim.trackers["1_moment"].result)
print(sim.trackers["1_moment"].counter)

[0.+0.j 0.+0.j]
[0 0]


only after a certain number of steps will the trajs be allowed to be observed. To (un)block the trajs, we call tracker.mark_equilibrated_trajs

In [25]:
for _ in tqdm(range(int(5e3))):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()

100%|██████████| 5000/5000 [00:08<00:00, 591.13it/s]


In [26]:
sim.trackers["1_moment"].equilibrated_trajs

array([ True,  True])

After computation, the result is stored in a buffer and the rolling stats are updated.

In [27]:
sim.trackers["1_moment"].compute()
sim.trackers["1_moment"].mark_equilibrated_trajs()
sim.trackers["1_moment"].equilibrated_trajs
print(f"Result buffer: {sim.trackers['1_moment'].result}")
print(f"Counter: {sim.trackers['1_moment'].counter}")
print(f"Rolling mean: {sim.trackers['1_moment'].rolling_mean}")
print(f"Rolling sq mean: {sim.trackers['1_moment'].rolling_sqr_mean_real}")
print()

Result buffer: [ 0.8512824 +0.0000000e+00j -0.27156106+3.3256638e-17j]
Counter: [1 1]
Rolling mean: [ 0.8512824 +0.0000000e+00j -0.27156106+3.3256638e-17j]
Rolling sq mean: [0.72468174 0.07374541]



The full loop looks someting like this

In [28]:
for _ in tqdm(range(sim.steps)):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()
        tr.compute()

print(f"Result buffer: {sim.trackers['1_moment'].result}")
print(f"Counter: {sim.trackers['1_moment'].counter}")
print(f"Rolling mean: {sim.trackers['1_moment'].rolling_mean}")
print(f"Rolling sq mean: {sim.trackers['1_moment'].rolling_sqr_mean_real}")
print()

100%|██████████| 10000/10000 [00:32<00:00, 310.65it/s]

Result buffer: [0.19667888+0.j 1.1239482 +0.j]
Counter: [10 10]
Rolling mean: [0.40872967+3.1654542e-16j 1.6160364 +1.7517607e-16j]
Rolling sq mean: [4.7432594 2.8339803]



To combine the rolling stats over different observables, call sim.finish()

In [29]:
sim.finish()
sim.trackers["1_moment"].rolling_mean

(2.024766+4.9172147e-16j)

from here stats can be calculated using utils.calculate_stats_complex

In [30]:
from src.utils import calculate_stats_complex
rolling_mean = sim.trackers["1_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["1_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["1_moment"].rolling_sqr_mean_imag
counter = sim.trackers["1_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"First moment: {np.round(mean, 5)} pm {np.round(sem_real, 5)}")

First moment: (0.10124+0j) pm 0.13576


The code is optimized for parallel execution of multiple trajectories

In [31]:
config = Config(dims = [1], sigma=1, interaction=1, trajs = int(1e4), steps = int(1e4), dt = 1e-3)
sim = ComplexLangevinSimulation(config)

sim.register_observable('1_moment', obs_kernel=n_moment_kernel, const_param={"order" : 1},  langevin_history=True, thermal_time=2, auto_corr=1.0)
sim.register_observable('2_moment', obs_kernel=n_moment_kernel, const_param={"order" : 2},  langevin_history=True, thermal_time=2, auto_corr=1.0)

for _ in tqdm(range(sim.steps)):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()
        tr.compute()
sim.finish()

100%|██████████| 10000/10000 [00:14<00:00, 673.89it/s]


In [32]:
# scipy routines
import numpy as np
from scipy.integrate import quad
from numba import jit

def n_moment(x, order):
    return x**order

# define rho and R
@jit
def action(x, sigma, lamb): return sigma/2*x**2+lamb/4*x**4

@jit
def rho(x, sigma, lamb): return np.exp(-action(x, sigma, lamb))

# define partition sums
def z_rho(sigma, lamb): 
    real_part = quad(lambda x: np.real(rho(x, sigma, lamb)), -np.inf, np.inf)[0]
    imag_part = quad(lambda x: np.imag(rho(x, sigma, lamb)), -np.inf, np.inf)[0]
    return real_part+1j*imag_part


# define exp val of n_moment wrt rho
@np.vectorize
def n_moment_rho(order, sigma, lamb):
    real_part = quad(lambda x: np.real(rho(x, sigma, lamb)*n_moment(x, order)), 
                     -np.inf, np.inf)[0]
    imag_part = quad(lambda x: np.imag(rho(x, sigma, lamb)*n_moment(x, order)), 
                     -np.inf, np.inf)[0]
    out = real_part + 1j*imag_part
    return out / z_rho(sigma, lamb)

In [33]:
sim.trackers["1_moment"].rolling_mean

(482.605+2.7351632e-12j)

In [37]:
rolling_mean = sim.trackers["1_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["1_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["1_moment"].rolling_sqr_mean_imag
counter = sim.trackers["1_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"First moment: in [{np.round(mean-sem_real, 5)}; {np.round(mean+sem_real, 5)}]")
print(f"scipy: {n_moment_rho(1, sim.sigma, sim.interaction)}")

First moment: in [(0.00361+0j); (0.00846+0j)]
scipy: 0j


In [52]:
rolling_mean = sim.trackers["2_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["2_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["2_moment"].rolling_sqr_mean_imag
counter = sim.trackers["2_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"Second moment: in[{np.round(mean-sem_real, 5).real}; {np.round(mean+sem_real, 5).real}] {np.round(sem_real/mean, 4).real*100}% relative error")
print(f"scipy: {n_moment_rho(2, sim.sigma, sim.interaction)}")

Second moment: in[0.46716; 0.47114] 0.42% relative error
scipy: (0.4679199169736833+0j)
